In [1]:
import os
import numpy as np
import datetime
import tensorflow as tf
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import LSTM
from keras.layers import GRU
from keras.layers import SimpleRNN
from keras.layers import SimpleRNNCell
from keras.layers import RNN
from keras.layers import Input
from keras.layers import Concatenate
from keras.layers import Reshape
from keras.activations import gelu
from keras.activations import relu
print(tf.__version__)

%load_ext tensorboard
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

# Data directory
dir = '/home/david/fst/autonomous-systems/src/control/learning_mpcc/AI/train_data/cm_clean_T1/'

model_id = 17
T = 1 # Previous timesteps
train_val_split = 0.75

if model_id == 1:
    modelName = 'model_1' # MLP 32
elif model_id == 2:
    modelName = 'model_2' # MLP 64
elif model_id == 3:
    modelName = 'model_3' # MLP 128
elif model_id == 4:
    modelName = 'model_4' # LSTM 32 + Linear
elif model_id == 5:
    modelName = 'model_5' # RNN
elif model_id == 6:
    modelName = 'model_6' # RNN + Linear
elif model_id == 7:
    modelName = 'model_7' # GRU + Linear
elif model_id == 8:
    modelName = 'model_8' # RNN + Dense32 + Dense 3
elif model_id == 9:
    modelName = 'model_9' # Dense 5-16 + Dense16-16 + RNN + Dense16-16 + Dense 16-5 + Linear
elif model_id == 10:
    modelName = 'model_10' # Dense 5-16 + Dense16-16 + LSTM + Dense16-16 + Dense 16-5 + Linear
elif model_id == 11:
    modelName = 'model_11' # Dense 5-64 + Dense64-32 + RNN + Dense32-64 + Dense 64-5 + Linear
elif model_id == 12:
    modelName = 'model_12' # Dense 5-64 + Dense64-32 + LSTM + Dense32-64 + Dense 64-5 + Linear
elif model_id == 13:
    modelName = 'model_13' # Same as model 12 but for T=5
elif model_id == 14:
    modelName = 'model_14' # Dense 5-32 + Dense 32-16 + RNN + Dense 16-32 + Dense 32-5 + Linear (5-3)
elif model_id == 15:
    modelName = 'model_15' # Dense 5-32 + Dense 32-16 + RNN + Linear (16-3)
elif model_id == 16:
    modelName = 'model_16' # Dense 5-32 + Dense 32-16 + RNN + Dense 16-32 + Dense 32-5 + Linear (5-3) with ReLU
elif model_id == 17:
    modelName = 'model_17' # Dense 5-32 + Dense 32-16 + RNN (with Cells) + Dense 16-32 + Dense 32-5 + Linear (5-3)
else:
    raise Exception('Invalid model_id')
    
x_train, x_test, y_train, y_test = [], [], [], []
# Load data
npz_files = [f for f in os.listdir(dir) if f.endswith(".npz")]
for file in npz_files:
    data = np.load(dir + file)
    x_train.append(data['x_train'])
    y_train.append(data['y_train'])
    x_test.append(data['x_test'])
    y_test.append(data['y_test'])
    
x_train = np.concatenate(x_train, axis=0)
y_train = np.concatenate(y_train, axis=0)
x_test = np.concatenate(x_test, axis=0)
y_test = np.concatenate(y_test, axis=0)

2023-05-25 16:16:27.733350: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


2.11.1


In [2]:
x_train, y_train = x_train[0:int(len(x_train)*train_val_split)], y_train[0:int(len(y_train)*train_val_split)]
x_val, y_val = x_train[int(len(x_train)*train_val_split):], y_train[int(len(y_train)*train_val_split):]

In [3]:
if model_id == 1:
    model = Sequential()
    model.add(Dense(32, input_dim=10, activation='softplus'))
    model.add(Dense(32, activation='softplus'))
    model.add(Dense(3, activation='softplus'))
elif model_id == 2:
    model = Sequential()
    model.add(Dense(64, input_dim=10, activation='softplus'))
    model.add(Dense(64, activation='softplus'))
    model.add(Dense(3, activation='softplus'))
elif model_id == 3:
    model = Sequential()
    model.add(Dense(128, input_dim=10, activation='softplus'))
    model.add(Dense(128, activation='softplus'))
    model.add(Dense(3, activation='softplus'))
elif model_id == 4:
    model = Sequential()
    model.add(LSTM(32,input_shape=(T+1,5)))
    # model.add(Dense(3,activation = 'softplus'))
    model.add(Dense(3))
elif model_id == 5:
    model = Sequential()
    model.add(SimpleRNN(3))
elif model_id == 6: 
    model = Sequential()
    model.add(SimpleRNN(5,name='SimpleRNN'))
    model.add(Dense(3,name='Dense'))
elif model_id == 7:
    model = Sequential()
    model.add(GRU(32,input_shape=(T+1,5)))
    model.add(Dense(3))
elif model_id == 8:
    model = Sequential()
    model.add(SimpleRNN(5))
    model.add(Dense(32))
    model.add(Dense(3))
elif model_id == 9:
    # Define the input shape
    input_shape = (2, 5)  # (num_timesteps, num_features)

    # Inputs
    inputs = Input(shape=input_shape, name='inputs')

    # Reshape the input
    input1 = inputs[:,0,:]
    input2 = inputs[:,1,:]
    
    # Fully Connected Layers
    fc1 = Dense(16, activation=gelu, name='fc1')
    fc2 = Dense(16, activation=gelu, name='fc2')
    
    #Process both inputs through FC layers
    x1 = fc1(input1)
    x1 = fc2(x1)
    
    x2 = fc1(input2)
    x2 = fc2(x2)
    
    # # Concatenate inputs
    # concat = Concatenate(axis=0, name='concat')([x1, x2])
    
    # Concatenate inputs
    concat = Concatenate(axis=1, name='concat')([x1, x2])

    # Reshape to make it 3D
    concat = Reshape((2, 16), name='reshape')(concat)
    
    # RNN
    rnn = SimpleRNN(16, name='rnn')(concat)
    
    # Reversed FC layers
    fc2_reversed = Dense(16, activation=gelu, name='fc2_reversed')
    fc1_reversed = Dense(5, activation=gelu, name='fc1_reversed')
    
    # Process RNN output through reversed FC layers
    x = fc2_reversed(rnn)
    x = fc1_reversed(x)
    
    # Output layer
    output = Dense(3, activation='linear',name='output')(x)
    
    # Define the model with the inputs and outputs
    model = tf.keras.Model(inputs=[inputs], outputs=output)
elif model_id == 10:
    # Define the input shape
    input_shape = (2, 5)  # (num_timesteps, num_features)

    # Inputs
    inputs = Input(shape=input_shape, name='inputs')

    # Reshape the input
    input1 = inputs[:,0,:]
    input2 = inputs[:,1,:]
    
    # Fully Connected Layers
    fc1 = Dense(16, activation=gelu, name='fc1')
    fc2 = Dense(16, activation=gelu, name='fc2')
    
    #Process both inputs through FC layers
    x1 = fc1(input1)
    x1 = fc2(x1)
    
    x2 = fc1(input2)
    x2 = fc2(x2)
    
    # # Concatenate inputs
    # concat = Concatenate(axis=0, name='concat')([x1, x2])
    
    # Concatenate inputs
    concat = Concatenate(axis=1, name='concat')([x1, x2])

    # Reshape to make it 3D
    concat = Reshape((2, 16), name='reshape')(concat)
    
    # RNN
    lstm = LSTM(16, name='rnn')(concat)
    
    # Reversed FC layers
    fc2_reversed = Dense(16, activation=gelu, name='fc2_reversed')
    fc1_reversed = Dense(5, activation=gelu, name='fc1_reversed')
    
    # Process RNN output through reversed FC layers
    x = fc2_reversed(lstm)
    x = fc1_reversed(x)
    
    # Output layer
    output = Dense(3, activation='linear',name='output')(x)
    
    # Define the model with the inputs and outputs
    model = tf.keras.Model(inputs=[inputs], outputs=output)
elif model_id == 11:
    # Define the input shape
    input_shape = (2, 5)  # (num_timesteps, num_features)

    # Inputs
    inputs = Input(shape=input_shape, name='inputs')

    # Reshape the input
    input1 = inputs[:,0,:]
    input2 = inputs[:,1,:]
    
    # Fully Connected Layers
    fc1 = Dense(64, activation=gelu, name='fc1')
    fc2 = Dense(32, activation=gelu, name='fc2')
    
    #Process both inputs through FC layers
    x1 = fc1(input1)
    x1 = fc2(x1)
    
    x2 = fc1(input2)
    x2 = fc2(x2)
    
    # # Concatenate inputs
    # concat = Concatenate(axis=0, name='concat')([x1, x2])
    
    # Concatenate inputs
    concat = Concatenate(axis=1, name='concat')([x1, x2])

    # Reshape to make it 3D
    concat = Reshape((2, 32), name='reshape')(concat)
    
    # RNN
    rnn = SimpleRNN(32, name='rnn')(concat)
    
    # Reversed FC layers
    fc2_reversed = Dense(64, activation=gelu, name='fc2_reversed')
    fc1_reversed = Dense(32, activation=gelu, name='fc1_reversed')
    
    # Process RNN output through reversed FC layers
    x = fc2_reversed(rnn)
    x = fc1_reversed(x)
    
    # Output layer
    output = Dense(3, activation='linear',name='output')(x)
    
    # Define the model with the inputs and outputs
    model = tf.keras.Model(inputs=[inputs], outputs=output, name='model_11')
elif model_id == 12:
    # Define the input shape
    input_shape = (2, 5)  # (num_timesteps, num_features)

    # Inputs
    inputs = Input(shape=input_shape, name='inputs')

    # Reshape the input
    input1 = inputs[:,0,:]
    input2 = inputs[:,1,:]
    
    # Fully Connected Layers
    fc1 = Dense(64, activation=gelu, name='fc1')
    fc2 = Dense(32, activation=gelu, name='fc2')
    
    #Process both inputs through FC layers
    x1 = fc1(input1)
    x1 = fc2(x1)
    
    x2 = fc1(input2)
    x2 = fc2(x2)
    
    # # Concatenate inputs
    # concat = Concatenate(axis=0, name='concat')([x1, x2])
    
    # Concatenate inputs
    concat = Concatenate(axis=1, name='concat')([x1, x2])

    # Reshape to make it 3D
    concat = Reshape((2, 32), name='reshape')(concat)
    
    # RNN
    lstm = LSTM(32, name='lstm')(concat)
    
    # Reversed FC layers
    fc2_reversed = Dense(64, activation=gelu, name='fc2_reversed')
    fc1_reversed = Dense(32, activation=gelu, name='fc1_reversed')
    
    # Process RNN output through reversed FC layers
    x = fc2_reversed(lstm)
    x = fc1_reversed(x)
    
    # Output layer
    output = Dense(3, activation='linear',name='output')(x)
    
    # Define the model with the inputs and outputs
    model = tf.keras.Model(inputs=[inputs], outputs=output, name='model_12')
elif model_id == 13:
    # Define the input shape
    input_shape = (6, 5)  # (num_timesteps, num_features)

    # Inputs
    inputs = Input(shape=input_shape, name='inputs')

    # Reshape the input
    input1 = inputs[:,0,:]
    input2 = inputs[:,1,:]
    input3 = inputs[:,2,:]
    input4 = inputs[:,3,:]
    input5 = inputs[:,4,:]
    input6 = inputs[:,5,:]
    
    # Fully Connected Layers
    fc1 = Dense(64, activation=gelu, name='fc1')
    fc2 = Dense(32, activation=gelu, name='fc2')
    
    #Process both inputs through FC layers
    x1 = fc1(input1)
    x1 = fc2(x1)
    
    x2 = fc1(input2)
    x2 = fc2(x2)
    
    x3 = fc1(input3)
    x3 = fc2(x3)
    
    x4 = fc1(input4)
    x4 = fc2(x4)
    
    x5 = fc1(input5)
    x5 = fc2(x5)
    
    x6 = fc1(input6)
    x6 = fc2(x6)
    
    # # Concatenate inputs
    # concat = Concatenate(axis=0, name='concat')([x1, x2])
    
    # Concatenate inputs
    concat = Concatenate(axis=1, name='concat')([x1, x2, x3, x4, x5, x6])

    # Reshape to make it 3D
    concat = Reshape((6, 32), name='reshape')(concat)
    
    # RNN
    lstm = LSTM(32, name='rnn')(concat)
    
    # Reversed FC layers
    fc2_reversed = Dense(64, activation=gelu, name='fc2_reversed')
    fc1_reversed = Dense(32, activation=gelu, name='fc1_reversed')
    
    # Process RNN output through reversed FC layers
    x = fc2_reversed(lstm)
    x = fc1_reversed(x)
    
    # Output layer
    output = Dense(3, activation='linear',name='output')(x)
    
    # Define the model with the inputs and outputs
    model = tf.keras.Model(inputs=[inputs], outputs=output, name='model_13')
elif model_id == 14:
    # Define the input shape
    input_shape = (2, 5)  # (num_timesteps, num_features)

    # Inputs
    inputs = Input(shape=input_shape, name='inputs')

    # Reshape the input
    input1 = inputs[:,0,:]
    input2 = inputs[:,1,:]
    
    # Fully Connected Layers
    fc1 = Dense(32, activation=gelu, name='fc1')
    fc2 = Dense(16, activation=gelu, name='fc2')
    
    #Process both inputs through FC layers
    x1 = fc1(input1)
    x1 = fc2(x1)
    
    x2 = fc1(input2)
    x2 = fc2(x2)
    
    # Concatenate inputs
    concat = Concatenate(axis=1, name='concat')([x1, x2])

    # Reshape to make it 3D
    concat = Reshape((2, 16), name='reshape')(concat)
    
    # RNN
    rnn = SimpleRNN(16, name='rnn')(concat)
    
    # Reversed FC layers
    fc2_reversed = Dense(32, activation=gelu, name='fc2_reversed')
    fc1_reversed = Dense(5, activation=gelu, name='fc1_reversed')
    
    # Process RNN output through reversed FC layers
    x = fc2_reversed(rnn)
    x = fc1_reversed(x)
    
    # Output layer
    output = Dense(3, activation='linear',name='output')(x)
    # output = Dense(3, activation=gelu,name='output')(x)
    
    # Define the model with the inputs and outputs
    model = tf.keras.Model(inputs=[inputs], outputs=output, name='model_14')
elif model_id == 15:
    # Define the input shape
    input_shape = (2, 5)  # (num_timesteps, num_features)

    # Inputs
    inputs = Input(shape=input_shape, name='inputs')

    # Reshape the input
    input1 = inputs[:,0,:]
    input2 = inputs[:,1,:]
    
    # Fully Connected Layers
    fc1 = Dense(32, activation=gelu, name='fc1')
    fc2 = Dense(16, activation=gelu, name='fc2')
    
    #Process both inputs through FC layers
    x1 = fc1(input1)
    x1 = fc2(x1)
    
    x2 = fc1(input2)
    x2 = fc2(x2)
    
    # Concatenate inputs
    concat = Concatenate(axis=1, name='concat')([x1, x2])

    # Reshape to make it 3D
    concat = Reshape((2, 16), name='reshape')(concat)
    
    # RNN
    rnn = SimpleRNN(16, name='rnn')(concat)
    
    # Output layer
    output = Dense(3, activation='linear',name='output')(rnn)
    # output = Dense(3, activation=gelu,name='output')(x)
    
    # Define the model with the inputs and outputs
    model = tf.keras.Model(inputs=[inputs], outputs=output, name='model_15')
elif model_id == 16:
    # Define the input shape
    input_shape = (2, 5)  # (num_timesteps, num_features)

    # Inputs
    inputs = Input(shape=input_shape, name='inputs')

    # Reshape the input
    input1 = inputs[:,0,:]
    input2 = inputs[:,1,:]
    
    # Fully Connected Layers
    fc1 = Dense(32, activation=relu, name='fc1')
    fc2 = Dense(16, activation=relu, name='fc2')
    
    #Process both inputs through FC layers
    x1 = fc1(input1)
    x1 = fc2(x1)
    
    x2 = fc1(input2)
    x2 = fc2(x2)
    
    # Concatenate inputs
    concat = Concatenate(axis=1, name='concat')([x1, x2])

    # Reshape to make it 3D
    concat = Reshape((2, 16), name='reshape')(concat)
    
    # RNN
    rnn = SimpleRNN(16, name='rnn')(concat)
    
    # Reversed FC layers
    fc2_reversed = Dense(32, activation=relu, name='fc2_reversed')
    fc1_reversed = Dense(5, activation=relu, name='fc1_reversed')
    
    # Process RNN output through reversed FC layers
    x = fc2_reversed(rnn)
    x = fc1_reversed(x)
    
    # Output layer
    output = Dense(3, activation='linear',name='output')(x)
    # output = Dense(3, activation=gelu,name='output')(x)
    
    # Define the model with the inputs and outputs
    model = tf.keras.Model(inputs=[inputs], outputs=output, name='model_16')
elif model_id == 17:
    # Define the input shape
    input_shape = (2, 5)  # (num_timesteps, num_features)

    # Inputs
    inputs = Input(shape=input_shape, name='inputs')

    # Reshape the input
    input1 = inputs[:,0,:]
    input2 = inputs[:,1,:]
    
    # Fully Connected Layers
    fc1 = Dense(32, activation=gelu, name='fc1')
    fc2 = Dense(16, activation=gelu, name='fc2')
    
    #Process both inputs through FC layers
    x1 = fc1(input1)
    x1 = fc2(x1)
    
    x2 = fc1(input2)
    x2 = fc2(x2)
    
    # Concatenate inputs
    concat = Concatenate(axis=1, name='concat')([x1, x2])

    # Reshape to make it 3D
    concat = Reshape((2, 16), name='reshape')(concat)
    
    # RNN
    rnn_cell = SimpleRNNCell(16, name= 'rnn_cell')
    rnn_layer = RNN(rnn_cell, name='rnn_layer')
    # h0 = [rnn_cell.get_initial_state(inputs=x1)]
    # h1 = rnn_layer(x1,h0)
    rnn = rnn_layer(concat)
    
    # Reversed FC layers
    fc2_reversed = Dense(32, activation=gelu, name='fc2_reversed')
    fc1_reversed = Dense(5, activation=gelu, name='fc1_reversed')
    
    # Process RNN output through reversed FC layers
    x = fc2_reversed(rnn)
    x = fc1_reversed(x)
    
    # Output layer
    output = Dense(3, activation='linear',name='output')(x)
    # output = Dense(3, activation=gelu,name='output')(x)
    
    # Define the model with the inputs and outputs
    model = tf.keras.Model(inputs=[inputs], outputs=output, name='model_17')

2023-05-25 16:16:30.265936: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:981] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-05-25 16:16:30.274764: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:981] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-05-25 16:16:30.274971: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:981] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-05-25 16:16:30.275809: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 FMA
To enable them in other operation

In [4]:
# Train model
model.compile(loss='mean_squared_error', optimizer='adam')

model.fit(x_train, y_train, epochs=100, batch_size=32, shuffle = True, verbose=2, validation_data=(x_val,y_val), callbacks=[tensorboard_callback])


Epoch 1/100


2023-05-25 16:16:34.375802: I tensorflow/compiler/xla/service/service.cc:173] XLA service 0x55b076297570 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2023-05-25 16:16:34.375831: I tensorflow/compiler/xla/service/service.cc:181]   StreamExecutor device (0): NVIDIA GeForce RTX 2080 with Max-Q Design, Compute Capability 7.5
2023-05-25 16:16:34.394415: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2023-05-25 16:16:34.554446: W tensorflow/compiler/xla/stream_executor/gpu/asm_compiler.cc:115] *** WARNING *** You are using ptxas 10.1.243, which is older than 11.1. ptxas before 11.1 is known to miscompile XLA code, leading to incorrect results or invalid-address errors.

You may not need to update to CUDA 11.1; cherry-picking the ptxas binary is often sufficient.
2023-05-25 16:16:34.598473: I tensorflow/compiler/jit/xla_compilation_cache.cc:4

49/49 - 5s - loss: 15.5262 - val_loss: 15.1076 - 5s/epoch - 96ms/step
Epoch 2/100
49/49 - 0s - loss: 10.4718 - val_loss: 6.1654 - 343ms/epoch - 7ms/step
Epoch 3/100
49/49 - 0s - loss: 3.1999 - val_loss: 1.6062 - 334ms/epoch - 7ms/step
Epoch 4/100
49/49 - 0s - loss: 1.7145 - val_loss: 1.2298 - 338ms/epoch - 7ms/step
Epoch 5/100
49/49 - 0s - loss: 1.4655 - val_loss: 1.0939 - 341ms/epoch - 7ms/step
Epoch 6/100
49/49 - 0s - loss: 1.3011 - val_loss: 1.0360 - 321ms/epoch - 7ms/step
Epoch 7/100
49/49 - 0s - loss: 1.1742 - val_loss: 0.7796 - 339ms/epoch - 7ms/step
Epoch 8/100
49/49 - 0s - loss: 1.0437 - val_loss: 0.6825 - 329ms/epoch - 7ms/step
Epoch 9/100
49/49 - 0s - loss: 0.9294 - val_loss: 0.6226 - 337ms/epoch - 7ms/step
Epoch 10/100
49/49 - 0s - loss: 0.8551 - val_loss: 0.5772 - 324ms/epoch - 7ms/step
Epoch 11/100
49/49 - 0s - loss: 0.8324 - val_loss: 0.7085 - 327ms/epoch - 7ms/step
Epoch 12/100
49/49 - 0s - loss: 0.8030 - val_loss: 0.5208 - 324ms/epoch - 7ms/step
Epoch 13/100
49/49 - 0s 

In [5]:
print(model.summary())
# Make predictions
trainPredictions = model.predict(x_train)
testPredictions = model.predict(x_test)

Model: "model_17"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 inputs (InputLayer)            [(None, 2, 5)]       0           []                               
                                                                                                  
 tf.__operators__.getitem (Slic  (None, 5)           0           ['inputs[0][0]']                 
 ingOpLambda)                                                                                     
                                                                                                  
 tf.__operators__.getitem_1 (Sl  (None, 5)           0           ['inputs[0][0]']                 
 icingOpLambda)                                                                                   
                                                                                           

In [6]:
# Evaluate model
trainScore = model.evaluate(x_train, y_train, verbose=0)
print('Train Score: %.2f MSE (%.2f RMSE)' % (trainScore, np.sqrt(trainScore)))
testScore = model.evaluate(x_test, y_test, verbose=0)
print('Test Score: %.2f MSE (%.2f RMSE)' % (testScore, np.sqrt(testScore)))

Train Score: 0.49 MSE (0.70 RMSE)
Test Score: 0.55 MSE (0.74 RMSE)


In [7]:
tf.saved_model.save(model,'saved_models/' + modelName + '_T_' + str(T))
# model.save('saved_models/' + modelName + '_T_' + str(T)')
model.save_weights('saved_weights/' + modelName + '_T_' + str(T) + '.h5')

INFO:tensorflow:Assets written to: saved_models/model_17_T_1/assets


In [8]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  1


In [9]:
dummy_x = np.array([[1, 1, 1, 1, 1],[2, 2, 2, 2, 2]])
dummy_x = np.expand_dims(dummy_x, axis=0)
dummy_y = model.predict(dummy_x)
print(dummy_y)

1/1 [==============================] - 0s 177ms/step
[[ 4.2419868  6.256059  -0.5924856]]
